## Why model acrylic polymers?

<div style="text-align: center;">
  <img src="images/acryl.jpg"
       alt="Acrylic glass"
       style="width: 400px; display: inline-block;">
</div>

Acrylic glass, commonly known as poly(methyl methacrylate) (PMMA), is a transparent thermoplastic used in applications such as windows, optical components, displays, protective barriers, and lightweight structural parts. Its combination of optical transparency, low density, dimensional stability, and mechanical rigidity makes it an important engineering polymer.

PMMA is also a useful model system for molecular simulation. Its amorphous structure illustrates how polymer-chain packing, conformational motion, temperature, and intermolecular interactions determine macroscopic material properties. Unlike a crystalline solid, an amorphous polymer does not possess a perfectly ordered lattice. Its mechanical response must therefore be obtained from a statistically representative, equilibrated polymer configuration.

In this hands-on exercise, the elastic properties of the equilibrated PMMA system will be estimated from fluctuations of the simulation-box dimensions. As discussed in the preceding lecture, these equilibrium fluctuations are related to the material’s elastic response through linear-response theory.

In [ ]:
# Import all necessary libraries

import os
import sys
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import detrend, savgol_filter

from scripts.utils import show_py3dmol
from scripts.utils import show_smiles
from scripts.utils import write_vmd_polymer
from scripts.utils import gen_cnt
from scripts.get_vdw_radius import get_molecular_surface_area

import MDAnalysis as mda

from scripts.run_lammps import run_lammps_background
from scripts.init_lammps import init_lammps
LAMMPS_EXE = init_lammps()

HOME_DIR = os.getcwd()

cols = [
    "step", "temp", "etotal", "ke", "pe", "press",
    "vol", "density", "lx", "ly", "lz",
    "pxx", "pyy", "pzz",
]

### Note about notebook saving warnings

During trajectory visualization, the notebook may sometimes show a warning that it could not be saved. This can happen when large trajectory outputs or interactive viewers are stored in the notebook.

You can usually ignore this warning. If it becomes annoying or prevents saving, click on the affected output cell and clear the output.


## Explore the molecular structures

First, visualize the monomers and tetramers of the different acrylic polymer systems. Compare the polymer backbones and side groups, and briefly describe the main structural differences between the systems. Consider how these differences may affect chain flexibility, intermolecular interactions, packing, and ultimately the material properties.

Use the `show_smiles` function to generate a 2D representation of a molecule from its SMILES string:

```python
smiles_string = "CC(=C)C(=O)OC"  # example: methyl methacrylate
show_smiles(smiles_string)

In [ ]:
# Monomers
methoxy_monomer  = "C=C(C)C(=O)OC"
hydroxyethyl_monomer = "C=C(C)C(=O)OCCO"
phenoxy_monomer  = "C=C(C)C(=O)Oc1ccccc1"

# Methyl methacrylate tetramer
methoxy_tetramer = (
    "CC(C)(C(=O)OC)"
    "CC(C)(C(=O)OC)"
    "CC(C)(C(=O)OC)"
    "CC(C)C(=O)OC"
)

# Hydroxyethyl methacrylate tetramer
hydroxyethyl_tetramer = (
    "CC(C)(C(=O)OCCO)"
    "CC(C)(C(=O)OCCO)"
    "CC(C)(C(=O)OCCO)"
    "CC(C)C(=O)OCCO"
)

# Phenyl methacrylate tetramer
phenoxy_tetramer = (
    "CC(C)(C(=O)Oc1ccccc1)"
    "CC(C)(C(=O)Oc1ccccc1)"
    "CC(C)(C(=O)Oc1ccccc1)"
    "CC(C)C(=O)Oc1ccccc1"
)

# Choose one to visualize
smiles_string = ".."

In [ ]:
smiles_string = methoxy_monomer # example: methyl methacrylate

show_smiles(smiles_string)

### Select and visualize a polyacrylate system

Crosslinked models are available for three different polyacrylates:

- `methoxy`
- `hydroxyethyl`
- `phenoxy`

Each model was constructed from a total of 2000 monomers. Choose one system by setting the `system` variable accordingly:

```python
system = "methoxy"
```

The corresponding LAMMPS data file is then loaded and prepared for visualization:

```python
data_file = Path("data_polyacrylate") / f"{system}.data"

write_vmd_polymer(data_file)
```

After running the cell, you can go to the **Launcher** tab in JupyterLab and open the **Desktop** environment. On the desktop, double-click the **VMD Polymer** icon to inspect the selected polymer system in VMD if you are interested.

In [ ]:
system = "hydroxyethyl"
data_file = Path("data_polyacrylate") / f"{system}.data"

write_vmd_polymer(data_file)

### Investigate the polymer chain lengths

The LAMMPS data file contains the bond topology of the polymer system. MDAnalysis can use this topology to identify connected fragments, where each fragment corresponds to one bonded molecule or polymer chain.

The following code loads the system and identifies the connected chains:

```python
u = mda.Universe(
    data_file,
    format="DATA",
    atom_style="id resid type charge x y z",
)

chains = u.atoms.fragments
```

Each element of `chains` is an MDAnalysis `AtomGroup`. The number of atoms in an individual chain can be accessed using:

```python
atom_subset.n_atoms
```

For example, the number of atoms in the first chain can be obtained from an `AtomGroup` with:

```python
first_chain = chains[0]
print(first_chain.n_atoms)
```

A list comprehension can be used to apply the same operation to every element of a collection and store the results in a new list:

```python
chain_sizes = [chain.n_atoms for chain in chains]
```

Each monomer contributes approximately 12 atoms to the model. Therefore, estimate the chain length in monomer units by dividing the number of atoms in each chain by 12 and rounding to the nearest integer:

```python
a = [round(b / x) for b in c]
```

Develop code to investigate the chain lengths in the system. Determine how many polymer chains are present, summarize their lengths, and visualize the resulting chain-length distribution.

In [ ]:
u = mda.Universe(
    data_file,
    format="DATA",
    atom_style="id resid type charge x y z",
)

chains = u.atoms.fragments

# check and update the monomer size for your system
# methoxy = 12
# hydroxyethyl = 17
# phenoxy = 8

monomer_size = 12

chain_sizes = [round(chain.n_atoms/monomer_size) for chain in chains]

print(chain_sizes)

### Inspect the chain-length distribution

Use
```python
plt.hist(x, bins=50)
``` 
to visualize the chain-length distribution, and discuss whether the resulting distribution is reasonable for the generated polymer structure.

In [ ]:
plt.hist(chain_sizes)
plt.show()

### Calculate elastic moduli from simulation-cell fluctuations

The polyacrylate systems were sampled for **10 ns** using a triclinic barostat. Consequently, the cell lengths $a$, $b$, and $c$, the cell angles $\alpha$, $\beta$, and $\gamma$, and the cell volume $V$ fluctuate throughout the trajectory. Within the linear-response framework discussed in the lecture, these equilibrium fluctuations can be used to estimate the elastic moduli.

<div style="text-align: center;">
  <img src="images/triclinic.png"
       alt="Triclinic Box"
       style="width: 250px; display: inline-block;">
</div>

The Young's modulus associated with fluctuations of a cell length $L_i$, where $L_i \in \{a,b,c\}$, is calculated as

$$
E_i =
\frac{k_\mathrm{B}T\,\langle L_i\rangle^2}
{\mathrm{Var}(L_i)\,\langle V\rangle}.
$$

The three directional Young's moduli are therefore obtained from the fluctuations of $a$, $b$, and $c$:

$$
E_a,\qquad E_b,\qquad E_c.
$$

The shear modulus associated with fluctuations of a cell angle $A_i$, where $A_i \in \{\alpha,\beta,\gamma\}$, is calculated as

$$
G_i =
\frac{k_\mathrm{B}T}
{\mathrm{Var}(A_i)\,\langle V\rangle}.
$$

The corresponding shear moduli are

$$
G_\alpha,\qquad G_\beta,\qquad G_\gamma.
$$

The angular fluctuations must be expressed in **radians** before evaluating this expression.

The bulk modulus is calculated from the volume fluctuations as

$$
K =
\frac{k_\mathrm{B}T\,\langle V\rangle}
{\mathrm{Var}(V)}.
$$

Here, $\langle\cdot\rangle$ denotes a trajectory average. The variances are evaluated using the sample variance, corresponding to `ddof=1` in NumPy.

The Boltzmann constant is given in SI units, so $k_\mathrm{B}T$ has units of joules. LAMMPS volumes in `units real` are reported in $\text{Å}^3$, meaning that the fluctuation expressions initially produce an energy density in $\mathrm{J}/\text{Å}^3$. The conversion

$$
1\ \mathrm{J}/\text{Å}^3 = 10^{21}\ \mathrm{GPa}
$$

is therefore applied using

```python
fvol = 1e21
```

Use `cella`, `cellb`, and `cellc` to calculate $E_a$, $E_b$, and $E_c$. Use `cellalpha`, `cellbeta`, and `cellgamma`, converted to radians, to calculate $G_\alpha$, $G_\beta$, and $G_\gamma$. Use `vol` to calculate the bulk modulus $K$.

In [ ]:
# Functions for calculating elastic moduli from equilibrium box fluctuations

fvol = 1e21                  # Convert J/Å³ to GPa
kB = 1.380649e-23            # Boltzmann constant in J/K
temperature = 300.0          # Temperature in K
kbT = kB * temperature       # Thermal energy in J


def youngs_modulus(length, volume):
    """Calculate Young's modulus in GPa from cell-length fluctuations."""
    return fvol * kbT * np.mean(length) ** 2 / (np.var(length, ddof=1) * np.mean(volume))


def shear_modulus(angle, volume):
    """Calculate the shear modulus in GPa from cell-angle fluctuations in radians."""
    return fvol * kbT / (np.var(angle, ddof=1) * np.mean(volume))


def bulk_modulus(volume):
    """Calculate the bulk modulus in GPa from volume fluctuations."""
    return fvol * kbT * np.mean(volume) / np.var(volume, ddof=1)

### Load the fluctuation data

The fluctuation files are stored in the `data_polyacrylate` directory and follow the naming convention `fluc_{system}.dat`, where `system` is the selected polyacrylate model. First, construct the path to the corresponding file using `Path`, and then read the data into a pandas `DataFrame`.

An f-string, written as `f"..."`, inserts the value of a variable into a string. Expressions placed inside `{}` are replaced by their current values. For example, if `system = "methoxy"`, then

```python
f"fluc_{system}.dat"
```

produces the filename `fluc_methoxy.dat`.

Python's `Path` class provides a convenient and platform-independent way to work with files and directories. The `/` operator joins directory and file names into a complete path.

```python
file = Path("folder_name") / "file_name"
```

Read the file using the following column names:

```python
data = pd.read_csv(
    fluc_file,
    comment="#",
    sep=r"\s+",
    names=[
        "step", "temp", "etotal", "ke", "pe", "press", "vol", "density",
        "cella", "cellb", "cellc", "cellalpha", "cellbeta", "cellgamma",
    ],
)
```

In [ ]:
fluc_file = Path("data_polyacrylate") / f"fluc_{system}.dat"

data = pd.read_csv(
    fluc_file,
    comment="#",
    sep=r"\s+",
    names=[
        "step", "temp", "etotal", "ke", "pe", "press", "vol", "density",
        "cella", "cellb", "cellc", "cellalpha", "cellbeta", "cellgamma",
    ],
)


### Calculate the elastic moduli

Use the fluctuation data to calculate the elastic moduli of the selected polyacrylate system. The bulk modulus can be obtained directly from the sampled cell volume:

```python
vol = data["vol"]

B = bulk_modulus(volume):
print(f"Bulk modulus: {B:.3f} GPa")
```

The cell angles in the data file are given in degrees. Before calculating a shear modulus, convert the corresponding angle fluctuations to radians using `np.deg2rad`:

```python
cellalpha = np.deg2rad(data["cellalpha"])
```

Use the cell lengths, converted cell angles, and volume fluctuations to calculate the elastic moduli in all spatial directions. Compare the directional values and discuss whether the system behaves isotropically.

In [ ]:
cella = data["cella"] 
vol = data["vol"]
Emod_a = youngs_modulus(cella, vol)
print(Emod_a)


### Inspect the simulation-cell fluctuations

Convert the simulation step to time in nanoseconds using `t = step / 1000000.0`. Then plot the three cell lengths, $a$, $b$, and $c$, as functions of time and check whether their fluctuations remain stable throughout the trajectory.

```python
plt.plot(t, cella)
plt.plot(t, ..)


plt.xlabel("Time")
plt.ylabel(r"Cell length / $\mathrm{\AA}$")
plt.show()
```

In [ ]:
step = data["step"]
t = step / 1000000.0

plt.plot(t, cella)

plt.ylabel(r"Cell length / $\mathrm{\AA}$")
plt.xlabel(r"Time / ns")
plt.show()

### Account for drift and analyze shorter time windows

The present model system is not yet fully cured and needs longer relaxation. Additionally, the viscous motion of the polymer chains strongly influences fluctuations in the simulation box. As a result, the simulation-cell dimensions may show a slow drift over time and the calculated elastic response may appear anisotropic. Since the fluctuation formulas assume sampling around a stationary equilibrium state, such long-term trends can strongly affect the calculated moduli.

First, select a shorter time interval by defining `t_start` and `t_stop`. A Boolean mask identifies all data points within this interval and can be used to subset each quantity.

```python
time = np.array(data["step"] / 1000000.0)

t_start = 0
t_stop = 5

mask = (time >= t_start) & (time <= t_stop)

vol_window = np.array(vol)[mask]
cella_window = np.array(cella)[mask]
cellb_window ...
```

Next, remove the linear drift from the selected cell dimensions while preserving their mean values:

```python
cella_detrended = detrend(cella_window, type="linear") + cella_window.mean()
cellb_detrended = detrend(cellb_window, type="linear") + cellb_window.mean()
cellc_detrended = detrend(cellc_window, type="linear") + cellc_window.mean()
```

Repeat the analysis for different time windows and compare the resulting elastic moduli. Plot the detrended quantities as functions of time to verify the selected sampling interval and assess the remaining drift. Determine how sensitive the calculated values and apparent anisotropy are to the selected time window and to the removal of drift.

In [ ]:
time = np.array(data["step"] / 1000000.0)

t_start = 6
t_stop = 8

mask = (time >= t_start) & (time <= t_stop)

vol_window = np.array(vol)[mask]
cella_window = np.array(cella)[mask]

vol_detrended = detrend(vol_window, type="linear") + vol_window.mean()
cella_detrended = detrend(cella_window, type="linear") + cella_window.mean()

Emod_a_detrended = youngs_modulus(cella_detrended, vol_detrended)

print(Emod_a_detrended)
